In [0]:
# =====================================
# SETUP
# =====================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, avg, count, when, lag, stddev
from pyspark.sql.window import Window

# Create Spark session
spark = SparkSession.builder.appName("IJC427_Assignment").getOrCreate()

In [0]:
# =====================================
# LOAD DATA
# =====================================
# Load ICLR Parquet dataset
iclr_df = spark.read.parquet("/Volumes/workspace/default/ijc427/iclr26v1.parquet")
print("ICLR Schema:")
iclr_df.printSchema()
iclr_df.show(5)

# Load OpenAlex JSON dataset
openalex_df = spark.read.json("/Volumes/workspace/default/ijc427/openalex.json")
print("OpenAlex Schema:")
openalex_df.printSchema()
openalex_df.show(5)

ICLR Schema:
root
 |-- year: long (nullable = true)
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- author_ids: string (nullable = true)
 |-- decision: string (nullable = true)
 |-- scores: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- labels: string (nullable = true)

+----+---------+--------------------+--------------------+--------------------+----------+--------------------+---------+--------------------+-----------------+
|year|       id|               title|            abstract|             authors|author_ids|            decision|   scores|            keywords|           labels|
+----+---------+--------------------+--------------------+--------------------+----------+--------------------+---------+--------------------+-----------------+
|2017|B1-Hhnslg|Prototyp

In [0]:
# =====================================
# DATA CLEANING & PREPARATION
# =====================================
# Standardise decision labels
iclr_df = iclr_df.withColumn(
    "decision_standard",
    when(col("decision").rlike("(?i)accept"), "accepted")
    .when(col("decision").rlike("(?i)reject"), "rejected")
    .otherwise("withdrawn")
)

# Explode scores and keywords for later analysis
scores_df = iclr_df.withColumn("score", explode("scores")).filter(col("score").isNotNull())
keywords_df = iclr_df.withColumn("keyword", explode("keywords")).filter(col("keyword").isNotNull())

In [0]:
# =====================================
# Q1 :Count papers by decision
# =====================================
decision_counts = iclr_df.groupBy("decision_standard").count().orderBy("decision_standard")
decision_counts.show()

+-----------------+-----+
|decision_standard|count|
+-----------------+-----+
|         accepted|11213|
|         rejected|17290|
|        withdrawn|27403|
+-----------------+-----+



In [0]:
# =====================================
# Q2 :Accepted papers with any score < 5
# =====================================
accepted_low_score = scores_df.filter((col("decision_standard")=="accepted") & (col("score") < 5))
count_accepted_low_score = accepted_low_score.select("id", "title").distinct().count()
print("Number of accepted papers with at least one score < 5:", count_accepted_low_score)

Number of accepted papers with at least one score < 5: 1514


In [0]:
# =====================================
# Q3: Top 10 papers by mean score
# =====================================
mean_scores_df = scores_df.groupBy("id", "year", "title").agg(
    avg("score").alias("mean_score"),
    stddev("score").alias("std_score")  # optional, shows score variability
)
top10_df = mean_scores_df.orderBy(col("mean_score").desc()).limit(10)
top10_df.select("id", "title", "year", "mean_score", "std_score").show()

+-----------+--------------------+----+-----------------+------------------+
|         id|               title|year|       mean_score|         std_score|
+-----------+--------------------+----+-----------------+------------------+
| u1cQYxRI1H|Scaling In-the-Wi...|2025|             10.0|               0.0|
|  Sy8gdB9xx|Understanding dee...|2017|9.666666666666666|0.5773502691896258|
| 6Mxhg9PtDE|Safety Alignment ...|2025|              9.5|0.9999999999999999|
| 88nT0j5jAn|Universal Few-sho...|2023|9.333333333333334|1.1547005383792517|
| LyJi5ugyJx|Simplifying, Stab...|2025|              9.2|1.0954451150103321|
| gc8QAQfXv6|Unlocking the Pow...|2025|              9.0|1.1547005383792517|
| WCRQFlji2q|Do I Know This En...|2025|              9.0|1.1547005383792517|
| aN4Jf6Cx69|The mechanistic b...|2024|              9.0|1.1547005383792517|
|b-ny3x071E5|Bootstrapped Meta...|2022|              9.0|1.1547005383792517|
| YrycTjllL0|BigCodeBench: Ben...|2025|              9.0|1.1547005383792517|

In [0]:
# =====================================
# Q4: Keywords surge
# =====================================
# Count keywords per year
keyword_counts = keywords_df.groupBy("year", "keyword").count()

# Identify keywords with <5 occurrences one year, >50 next year
w = Window.partitionBy("keyword").orderBy("year")
keyword_changes = keyword_counts.withColumn("prev_count", lag("count").over(w))
keyword_surge = keyword_changes.filter((col("prev_count") < 5) & (col("count") > 50))

# Show top 10 surging keywords
keyword_surge.select("keyword", "year", "prev_count", "count").orderBy(col("count").desc()).limit(10).show()

+--------------------+----+----------+-----+
|             keyword|year|prev_count|count|
+--------------------+----+----------+-----+
|   test-time scaling|2026|         3|   87|
|                 llm|2024|         4|   76|
|large reasoning m...|2026|         1|   57|
|   spatial reasoning|2026|         3|   57|
+--------------------+----+----------+-----+



In [0]:
# =====================================
#Q5: ICLR papers in OpenAlex
# =====================================
joined_df = iclr_df.join(openalex_df, iclr_df.title == openalex_df.title, "inner")

# Group by year, count, and order by year
yearly_counts = joined_df.groupBy("year").count().orderBy(col("year").asc())

# Show the sorted result
yearly_counts.show()

+----+-----+
|year|count|
+----+-----+
|2017|  154|
|2018|  309|
|2019|  199|
|2020|  433|
|2021|  423|
|2022|   52|
|2023|   19|
|2024|    1|
|2025|    2|
|2026|    1|
+----+-----+



In [0]:
# =====================================
# Q6: High-citation, low-score papers
# =====================================
# Filter low-score papers
low_score_df = mean_scores_df.filter(col("mean_score") < 5)

# Join with OpenAlex for citation info
joined_low = low_score_df.join(openalex_df, "id")

# Only proceed if matches exist
if joined_low.count() > 0:
    # Compute 95th percentile of citations
    citation_95 = joined_low.approxQuantile("cited_by_count", [0.95], 0.01)[0]
    print(f"95th percentile citation count: {citation_95}")
    
    high_citation_df = joined_low.filter(col("cited_by_count") >= citation_95)
    high_citation_df.select("id", "title", "year", "mean_score", "cited_by_count").show()
else:
    print("No papers found with mean_score < 5 and matching OpenAlex entries.")

No papers found with mean_score < 5 and matching OpenAlex entries.
